# Lab 2: Knowledge Graphs for RAG & Agents

**Session 2 · Lab 2 of 3 · ~35 minutes**

## Goal

Extract entity-relationship triplets from financial documents using `ai_extract`,
store them as Delta tables, and explore the resulting knowledge graph using both
SQL and Genie's natural language interface.

## What you'll do

1. Extract entity-relationship triplets from earnings call transcripts
2. Build `entities` and `relationships` Delta tables
3. Query the graph using multi-hop SQL traversal
4. Explore the graph using Genie natural language queries
5. Create a reusable graph lookup function for use in agents

---

In [0]:
%run ../utils/config

In [0]:
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print(f"✓ Default catalog.schema set to {catalog}.{schema}")

## Part 1: Extract entity-relationship triplets

We'll use `ai_extract` to extract structured entity and relationship data from
earnings call transcripts — the richest source of named entity mentions.

A **triplet** is: `(subject entity) → [predicate relationship] → (object entity)`

Example: `NVIDIA → partners_with → Microsoft`

In [0]:
print(f"\nReading source data from {catalog}.{shared_schema}.financial_docs_silver...")

source_df = spark.table(f"{shared_schema}.financial_docs_silver").where(f"document_type ILIKE '%Earnings%'").orderBy("company", "fiscal_period")

source_count = source_df.count()
print(f"✓ Loaded {source_count} records for entity extraction")
display(source_df)

### Helper functions

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def escape_sql_literal(value: str) -> str:
    """Escape string for SQL"""
    return value.replace("\\", "\\\\").replace("'", "\\'").replace('"', '\\"')

### Define the entity extraction prompt
The function `ai_query()` gives us greater flexibility than `ai_extract()` for this task as we can provide a richer prompt to describe more precisely how we want to extract the entities.

In [0]:
ENTITY_EXTRACTION_PROMPT = """
Extract entity-relationship triplets from this financial earnings call transcript.

Entities: Identify all named entities (companies, people, products, technologies, markets, regulations).
  - name: Entity name
  - entity_type: One of Company, Person, Product, Technology, Market, Regulation
  - description: Brief description of this entity in context
Relationships: Identify how entities relate to each other.
  - subject: Subject entity name
  - predicate: One of partners_with, competes_with, invests_in, acquires, uses_technology, supplies_to, mentions
  - object: Object entity name

TEXT TO ANALYZE:
"""

In [0]:
import json

# Use JSON schema format (not STRUCT) so we can set additionalProperties: false,
# which the OpenAI-based model endpoints require for structured output.
response_schema = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "extraction",
        "schema": {
            "type": "object",
            "properties": {
                "data": {
                    "type": "object",
                    "properties": {
                        "entities": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "name": {"type": "string"},
                                    "entity_type": {"type": "string"},
                                    "description": {"type": "string"}
                                },
                                "required": ["name", "entity_type", "description"],
                                "additionalProperties": False
                            }
                        },
                        "relationships": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "subject": {"type": "string"},
                                    "predicate": {"type": "string"},
                                    "object": {"type": "string"}
                                },
                                "required": ["subject", "predicate", "object"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["entities", "relationships"],
                    "additionalProperties": False
                }
            },
            "required": ["data"],
            "additionalProperties": False
        },
        "strict": True
    }
})

### Process records to extract entities
Use the `ai_query()` AI function to extract entities and relationships from the parsed text in the `financial_docs_silver` table.

In [0]:
# For models requiring strict JSON
from pyspark.sql.functions import expr
import json

MODEL = 'databricks-gpt-5-nano'
escaped_prompt = escape_sql_literal(ENTITY_EXTRACTION_PROMPT)

# Extract entity-relationship triplets using ai_query with structured output
# We run this on earnings call transcripts — they contain the richest relationship mentions

# Apply the LLM inference to each row, concatenating the prompt, the review text, and the tail string.
triplets_df = source_df.withColumn(
    "result",
    expr(f"""
         ai_query(
             '{MODEL}', 
             request => concat('{escaped_prompt}', plain_text),
             responseFormat => '{response_schema}')
        """
    )
)

# Persist to Delta so downstream cells don't re-trigger ai_query
# (.cache() is not supported on serverless compute)
triplets_df.write.mode("overwrite").saveAsTable("triplets_raw")
triplets_df = spark.table("triplets_raw")

print(f"✓ Extracted triplets from {triplets_df.count()} earnings call transcripts")
display(triplets_df)

## Part 2: Build the graph tables

Explode the nested arrays into flat `entities` and `relationships` tables in your personal schema.

In [0]:
from pyspark.sql.functions import col, explode, from_json, row_number, concat, lpad, lit
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# Define schema matching the ai_query responseFormat
entity_type = ArrayType(StructType([
    StructField("name", StringType()), StructField("entity_type", StringType()), StructField("description", StringType())
]))
rel_type = ArrayType(StructType([
    StructField("subject", StringType()), StructField("predicate", StringType()), StructField("object", StringType())
]))
result_schema = StructType([
    StructField("data", StructType([
        StructField("entities", entity_type),
        StructField("relationships", rel_type)
    ]))
])

# Parse the cached JSON result column
parsed_df = triplets_df.withColumn("parsed", from_json(col("result"), result_schema))

# Build the entities table
entities_df = (
    parsed_df
    .select("company", "fiscal_period", explode(col("parsed.data.entities")).alias("entity"))
    .select(
        "company",
        "fiscal_period",
        col("entity.name").alias("entity_name"),
        col("entity.entity_type").alias("entity_type"),
        col("entity.description").alias("description"),
    )
    .dropDuplicates(["entity_name", "entity_type"])
)

# Add UID column
window_spec = Window.partitionBy("entity_type").orderBy("entity_name")
entities_df = entities_df.withColumn(
    "UID",
    concat(
        col("entity_type"),
        lit("_"),
        lpad(row_number().over(window_spec).cast("string"), 3, "0")
    )
)

entities_df.write.mode("overwrite").saveAsTable("entities")
print(f"\u2713 entities table: {spark.table('entities').count():,} unique entities")
display(entities_df)

In [0]:
# Build the relationships table from the same parsed result
relationships_df = (
    parsed_df
    .select("company", "fiscal_period", explode(col("parsed.data.relationships")).alias("rel"))
    .select(
        "company",
        "fiscal_period",
        col("rel.subject").alias("subject"),
        col("rel.predicate").alias("predicate"),
        col("rel.object").alias("object"),
    )
    .dropDuplicates(["subject", "predicate", "object"])
)

relationships_df.write.mode("overwrite").saveAsTable("relationships")
print(f"\u2713 relationships table: {spark.table('relationships').count():,} unique relationships")
display(relationships_df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, concat, lpad, lit

# There are likely objects of relationships that don't exist as entities.
missing_entities_df = spark.sql("""
    SELECT DISTINCT
        r.object                           AS entity_name,
        'Unknown'                          AS entity_type
    FROM relationships r
    LEFT ANTI JOIN entities e
      ON r.object = e.entity_name
      AND r.company = e.company
      AND r.fiscal_period = e.fiscal_period
    WHERE r.object IS NOT NULL
""")

window_spec = Window.partitionBy("entity_type").orderBy("entity_name")
missing_entities_df = (missing_entities_df
    .withColumn(
        "UID",
        concat(
            col("entity_type"),
            lit("_"),
            lpad(row_number().over(window_spec).cast("string"), 3, "0")
        ))
    .withColumn("company", lit("Unknown"))
    .withColumn("fiscal_period", lit("Unknown"))
    .withColumn("description", lit("Identified through relationships")))

# Reorder columns to match the entities table schema (insertInto uses positional order)
missing_entities_df = missing_entities_df.select(
    "company", "fiscal_period", "entity_name", "entity_type", "description", "UID"
)

# Insert missing entities into the existing entities table
missing_entities_df.write.mode("append").insertInto("entities")

print(f"\u2713 wrote {missing_entities_df.count():,} additional unique entities based on relationship objects")
display(missing_entities_df)

## Part 3: Query the graph

Graph traversal in Delta tables uses self-joins to follow relationship chains.

In [0]:
# Preview the entity types extracted
display(spark.sql("""
    SELECT entity_type, COUNT(*) AS count
    FROM entities
    GROUP BY entity_type
    ORDER BY count DESC
"""))

In [0]:
# Preview the relationship types
display(spark.sql("""
    SELECT predicate, COUNT(*) AS count
    FROM relationships
    GROUP BY predicate
    ORDER BY count DESC
"""))

In [0]:
# What technology did Amazon mention that they use?
display(spark.sql("""
    SELECT subject, predicate, object, fiscal_period
    FROM relationships
    WHERE subject ilike 'Amazon%'
      AND predicate ilike 'uses_technology'
    ORDER BY fiscal_period DESC
"""))

In [0]:
# Multi-hop: Who are the partners OF Amazon's partners? (2-hop traversal)
display(spark.sql("""
    SELECT
        r1.subject  AS company,
        r1.predicate AS hop1_rel,
        r1.object   AS partner,
        r2.predicate AS hop2_rel,
        r2.object   AS partners_partner
    FROM relationships r1
    JOIN relationships r2
      ON r1.object = r2.subject
     AND r2.predicate IN ('partners_with', 'invests_in')
    WHERE r1.subject = 'NVIDIA'
      AND r1.predicate = 'partners_with'
      AND r2.object != 'NVIDIA'  -- Avoid circular paths
    ORDER BY partner, partners_partner
    LIMIT 20
"""))

In [0]:
# Which AI-related entities are mentioned most by across ALL companies?
display(spark.sql("""
    SELECT
        object AS ai_entity,
        COUNT(DISTINCT subject) AS mentioned_by_n_companies,
        COLLECT_SET(subject) AS mentioned_by
    FROM relationships
    WHERE predicate IN ('partners_with', 'invests_in', 'uses_technology')
    GROUP BY object
    HAVING COUNT(DISTINCT subject) >= 3
    ORDER BY mentioned_by_n_companies DESC
    LIMIT 20
"""))

In [0]:
# How does the Microsoft → OpenAI relationship emerge from the documents?
display(spark.sql("""
    SELECT DISTINCT subject, predicate, object, company AS mentioned_in, fiscal_period
    FROM relationships
    WHERE (subject = 'Microsoft' AND object = 'OpenAI')
       OR (subject = 'OpenAI'   AND object = 'Microsoft')
    ORDER BY fiscal_period
"""))

In [0]:
display(spark.sql("""
    SELECT
        e.entity_name AS chief_person,
        e.description AS title,
        r.predicate,
        r.object AS mentioned_entity
    FROM entities e
    JOIN relationships r
      ON e.entity_name = r.subject
    WHERE e.entity_type = 'Person'
      AND LOWER(e.description) LIKE '%chief%'
"""))

TODO: Need instructions on how to create this Genie Space

also

TODO: come up with questions that Genie will actually answer well

## Part 4: Query with Genie

Genie lets non-technical analysts query the knowledge graph using natural language.

**In the UI:**
1. Navigate to **Databricks SQL** → **Genie**
2. Open the **Financial Intelligence Graph Space**
3. Try these questions in natural language:
   - *"Which companies mentioned the same AI chip suppliers?"*
   - *"Show me companies that NVIDIA competes with"*
   - *"What technologies do all 5 companies mention investing in?"*
   - *"Which entities appear in both Microsoft and Meta earnings calls?"*

Genie translates your question into SQL against the `entities` and `relationships` tables.

In [0]:
# Run the equivalent SQL to verify Genie's output
display(spark.sql("""
    -- "Which entities appear in both Microsoft and Meta earnings calls?"
    SELECT object AS shared_entity, COUNT(DISTINCT subject) AS company_count
    FROM relationships
    WHERE subject IN ('Microsoft', 'Meta')
    GROUP BY object
    HAVING COUNT(DISTINCT subject) = 2
    ORDER BY shared_entity
"""))

## Part 5: Create a graph lookup function for agents

This function wraps the graph tables in a simple interface that an AI agent can use as a **tool**.

In [0]:
def get_entity_relationships(entity_name, predicate_filter=None, limit=20):
    """Look up all relationships for a named entity.
    
    Args:
        entity_name:      Entity to look up (e.g. 'NVIDIA', 'OpenAI')
        predicate_filter: Optional relationship type to filter on (e.g. 'partners_with')
        limit:            Maximum number of relationships to return
    
    Returns:
        List of (subject, predicate, object) tuples involving this entity
    """
    predicate_clause = f"AND predicate = '{predicate_filter}'" if predicate_filter else ""
    
    results = spark.sql(f"""
        SELECT subject, predicate, object
        FROM relationships
        WHERE (subject = '{entity_name}' OR object = '{entity_name}')
        {predicate_clause}
        ORDER BY predicate, subject
        LIMIT {limit}
    """).collect()
    
    return [(r['subject'], r['predicate'], r['object']) for r in results]


# Test it
nvidia_relationships = get_entity_relationships("NVIDIA", predicate_filter="uses_technology")
print(f"NVIDIA 'partners_with' relationships:")
for s, p, o in nvidia_relationships[:10]:
    print(f"  {s} → {p} → {o}")

In [0]:
edge_df = spark.sql("""
    WITH subject_ids AS (
        SELECT 
            e.entity_type AS node_start_key,
            r.subject,
            ROW_NUMBER() OVER (PARTITION BY e.entity_type ORDER BY r.subject) AS rn
        FROM relationships r
        LEFT JOIN entities e
          ON r.subject = e.entity_name
        GROUP BY e.entity_type, r.subject
    ),
    object_ids AS (
        SELECT 
            e.entity_type AS node_end_key,
            r.object,
            ROW_NUMBER() OVER (PARTITION BY e.entity_type ORDER BY r.object) AS rn
        FROM relationships r
        LEFT JOIN entities e
          ON r.object = e.entity_name
        GROUP BY e.entity_type, r.object
    )

    SELECT 
        CONCAT(e.entity_type, '_', LPAD(sid.rn, 3, '0')) AS node_start_id,
        e.entity_type AS node_start_key,
        r.subject, 
        r.predicate AS relationship, 
        r.object,
        CONCAT(oe.entity_type, '_', LPAD(oid.rn, 3, '0')) AS node_end_id,
        COALESCE(oe.entity_type, 'entity') AS node_end_key,
        '{"_label": "' || r.subject || '"}' AS node_start_properties,
        '{"_label": "' || r.object || '"}' AS node_end_properties


    FROM relationships r
    LEFT JOIN entities e
      ON r.subject = e.entity_name
    LEFT JOIN subject_ids sid
      ON r.subject = sid.subject AND e.entity_type = sid.node_start_key
    LEFT JOIN entities oe
      ON r.object = oe.entity_name
    LEFT JOIN object_ids oid
      ON r.object = oid.object AND oe.entity_type = oid.node_end_key
    ORDER BY r.subject, relationship, r.object
""")


edge_df.write.mode("overwrite").saveAsTable("entity_edges")
display(edge_df)

In [0]:
# This function can be wrapped as an MLflow tool for use in an AI agent:
# 
# @mlflow.pyfunc.tool
# def graph_lookup(entity_name: str, predicate: str = None) -> list:
#     """Look up entity relationships in the financial knowledge graph.
#     Use this tool when you need to find how companies, products, or people
#     are related to each other based on their financial disclosures.
#     """
#     return get_entity_relationships(entity_name, predicate)
#
# Combined with Vector Search retrieval, this tool enables an agent to:
# 1. Semantically retrieve relevant document chunks
# 2. Identify entities mentioned in those chunks
# 3. Traverse the graph to find related entities not explicitly in the chunks
# 4. Return a richer, more connected answer

print("ℹ️  See comments above for how to integrate graph_lookup as an agent tool")

## Key Takeaways

1. **Knowledge graphs on Databricks** = entity/relationship Delta tables — no graph database required
2. **`ai_extract`** with nested array schemas is the practical way to extract triplets at scale
3. **Multi-hop traversal** in SQL reveals non-obvious connections ("who are NVIDIA's partners' partners?")
4. **Genie** makes the graph accessible to non-technical analysts without writing SQL
5. **Graph lookup functions** are the building block for graph-augmented AI agents

---

**Next:** Lab 3 — Designing hybrid context layers

→ Open `04_hybrid_context_layers`